# V1 - Learning the Basics

In [ ]:
!pip install -q anthropic openai google-generativeai
from google.colab import userdata

# === Choose your LLM provider ===
# Pick one. Put that provider's API key in Colab's Secrets tab (key icon, left
# sidebar) under the secret name shown in PROVIDER_CONFIG below.
PROVIDER = "anthropic"   # "gemini" | "anthropic" | "deepseek" | "groq"

PROVIDER_CONFIG = {
    "gemini":    {"secret": "GOOGLE_API_KEY",    "model": "gemini-flash-latest"},
    "anthropic": {"secret": "ANTHROPIC_API_KEY", "model": "claude-haiku-4-5"},
    "deepseek":  {"secret": "DEEPSEEK_API_KEY",  "model": "deepseek-chat",
                  "base_url": "https://api.deepseek.com"},
    "groq":      {"secret": "GROQ_API_KEY",      "model": "llama-3.3-70b-versatile",
                  "base_url": "https://api.groq.com/openai/v1"},
}
# Model IDs change over time — if a call 404s, check the provider's console for
# the current name and update the "model" value above.

_cfg = PROVIDER_CONFIG[PROVIDER]
_api_key = userdata.get(_cfg["secret"])

if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_api_key)
    _gemini_model = genai.GenerativeModel(_cfg["model"])
elif PROVIDER == "anthropic":
    import anthropic
    _anthropic = anthropic.Anthropic(api_key=_api_key)
else:  # deepseek / groq are OpenAI-compatible — same SDK, different base_url
    from openai import OpenAI
    _openai = OpenAI(api_key=_api_key, base_url=_cfg["base_url"])

def generate(prompt, system=None, max_tokens=4096):
    """Provider-agnostic text generation — returns the model's text response.
    Grounding/instructions go in `system`, the task in `prompt`. The rest of the
    notebook only calls generate(), so switching providers means editing PROVIDER."""
    if PROVIDER == "gemini":
        full = f"{system}\n\n{prompt}" if system else prompt
        return _gemini_model.generate_content(full).text
    if PROVIDER == "anthropic":
        kwargs = {"model": _cfg["model"], "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        return _anthropic.messages.create(**kwargs).content[0].text
    # deepseek / groq — OpenAI-compatible chat completions
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
    return _openai.chat.completions.create(
        model=_cfg["model"], messages=messages, max_tokens=max_tokens
    ).choices[0].message.content

Pick your LLM provider in the cell above (`PROVIDER`) and add that provider's API key to Colab's Secrets tab. Everything below calls a single, provider-agnostic `generate()` function, so the rest of the notebook doesn't change when you switch providers.

In [ ]:
job_description = input("Paste job description: ")  # or just a triple-quoted string cell for pasting long text
resume = input("Paste resume: ")
github_username = input("GitHub username (optional): ")

Paste job description: Job description At NiCE, we don’t limit our challenges. We challenge our limits. Always. We’re ambitious. We’re game changers. And we play to win. We set the highest standards and execute beyond them. And if you’re like us, we can offer you the ultimate career opportunity that will light a fire within you.  So, what’s the role all about?  This is not just another full stack role—this is a chance to help modernize a core NICE platform at scale. You’ll join a high-visibility engineering team driving the transformation of a legacy UI (ASPX/.NET) into modern Angular/React front ends powered by microservices architecture.  You’ll be part of the Novus team, a fast-moving, highly collaborative group at the center of NICE’s innovation efforts—working closely with engineers, architects, and cross-functional partners.  Even more exciting: this team is actively leveraging AI tools to accelerate development, automate migrations, and rethink how software is built—giving you h

Some simple python to prompt the user for a few items to test out our system

In [ ]:
import requests
repos = []
if github_username:
    r = requests.get(f"https://api.github.com/users/{github_username}/repos", timeout=10)
    data = r.json()
    if r.status_code == 200 and isinstance(data, list):
        repos = [{"name": x["name"], "desc": x.get("description"), "lang": x.get("language")} for x in data]
    else:
        # A bad username returns a dict like {"message": "Not Found"}, not a list —
        # iterating that would crash, so degrade gracefully instead.
        print(f"⚠️ Could not fetch repos for '{github_username}' (HTTP {r.status_code}). Continuing without GitHub data.")

Optional Call to Github API to fetch repos that may or may not be a fit

In [ ]:
print(f"Using provider: {PROVIDER}  |  model: {_cfg['model']}")

In [ ]:
prompt = f"""
You are a resume tailoring assistant...
JOB DESCRIPTION: {job_description}
RESUME: {resume}
GITHUB PROJECTS: {repos}

Return two sections:
1. TAILORED RESUME (markdown)
2. WHAT CHANGED AND WHY (bulleted)
"""
print(generate(prompt))

# V1.1 - Improving the Foundation

*(Interim step between V1 and V2 — summarized here rather than shown as separate cells, since it reuses V1's single-prompt approach.)*

V1.1 keeps V1's one-shot prompt but hardens the input/output that a live workshop kept tripping over:

- **Job description by URL** — fetch and strip the page with BeautifulSoup, not just pasted text.
- **Resume by file upload** — accept a `.md`/`.txt` upload, not just pasted text.
- **Graceful GitHub failures** — wrap the repo fetch so a bad username or rate limit degrades to "no repo evidence" instead of crashing.

The next notebook (**V2**) rebuilds this foundation with production patterns: real RAG (chunking, embeddings, retrieval, evaluation), a multi-tool agent, and AI safety guardrails.